# Домашнее задание 5. Проверка данных и обработка ошибок

В работе восемь заданий. Вы напишете небольшие функции, которые проверяют файлы,
перехватывают ошибки чтения и не допускают диагностику данных с неверной структурой
или частотой дискретизации.

Заменяйте `...` своим кодом и запускайте ячейку `check` после каждого задания.
Имена функций и результирующих переменных изменять не нужно.

## Подготовка

In [1]:
from pathlib import Path
from scipy.io import loadmat, wavfile

DATA = Path("data")
EXPECTED_FS = 10_000

In [2]:
def check(task, value):
    try:
        if task == 1:
            ok = value is True
        elif task == 2:
            ok = value == (True, False)
        elif task == 3:
            ok = value == (True, False)
        elif task == 4:
            expected = [
                "Идеальное состояние", "Идеальное состояние",
                "Хорошее состояние", "Средний износ",
                "Сильный износ", "Критический износ",
            ]
            ok = value == expected
        elif task == 5:
            ok = value == "Файл data/Подшипник_0.wav не найден."
        elif task == 6:
            ok = value == "Файл некорректен."
        elif task == 7:
            ok = value == "Частота дискретизации аудио не соответствует модели."
        elif task == 8:
            ok = value == "Данные готовы к диагностике"
        else:
            raise ValueError("неизвестный номер задания")
    except Exception as exc:
        print(f"❌ Задание {task}: {type(exc).__name__}: {exc}")
        return
    print(f"{'✅' if ok else '❌'} Задание {task}" +
          (" выполнено" if ok else ": проверьте результат"))

## Задание 1. Существует ли файл

Напишите функцию `check_file(path)`, которая возвращает `True`, если путь указывает
на существующий файл, и `False` в противном случае. Проверьте `Bearing.mat`.

In [3]:
# ✍ Задание 1
def check_file(path):
    return Path(path).is_file()

ans1 = check_file(DATA / "Bearing.mat")
print(ans1)

True


In [4]:
check(1, ans1)

✅ Задание 1 выполнено


## Задание 2. Безопасное чтение WAV

Напишите `handle_exceptions(path)`. Функция должна попытаться прочитать файл через
`wavfile.read` и вернуть `True`. При любой ошибке чтения верните `False`.
Проверьте корректный и повреждённый файлы.

In [5]:
# ✍ Задание 2
def handle_exceptions(path):
    try:
        wavfile.read(path)
        return True
    except Exception:
        return False

ans2 = (
    handle_exceptions(DATA / "Подшипник_1.wav"),
    handle_exceptions(DATA / "Подшипник_.wav"),
)
print(ans2)

(True, False)


In [6]:
check(2, ans2)

✅ Задание 2 выполнено


## Задание 3. Структура MAT-файла

Напишите `check_structure(path)`. Загрузите MAT-файл и верните `True`, только если
в нём присутствуют все четыре ключа: `normal`, `roller`, `inner`, `outer`.
Для ошибки чтения верните `False`.

In [7]:
# ✍ Задание 3
def check_structure(path):
    try:
        data = loadmat(path)
        required = {"normal", "roller", "inner", "outer"}
        return required.issubset(data.keys())
    except Exception:
        return False

ans3 = (
    check_structure(DATA / "Bearing.mat"),
    check_structure(DATA / "Bearing_0.mat"),
)
print(ans3)

(True, False)


In [8]:
check(3, ans3)

✅ Задание 3 выполнено


## Задание 4. Категория состояния

Напишите функцию `classify_condition(value)` с границами:

- до 10 включительно — `Идеальное состояние`;
- до 20 включительно — `Хорошее состояние`;
- до 50 включительно — `Средний износ`;
- до 75 включительно — `Сильный износ`;
- выше 75 — `Критический износ`.

In [9]:
# ✍ Задание 4
def classify_condition(value):
    if value <= 10:
        return "Идеальное состояние"
    elif value <= 20:
        return "Хорошее состояние"
    elif value <= 50:
        return "Средний износ"
    elif value <= 75:
        return "Сильный износ"
    return "Критический износ"

ans4 = [classify_condition(x) for x in [0, 10, 20, 50, 75, 76]]
print(ans4)

['Идеальное состояние', 'Идеальное состояние', 'Хорошее состояние', 'Средний износ', 'Сильный износ', 'Критический износ']


In [10]:
check(4, ans4)

✅ Задание 4 выполнено


## Задание 5. Единая проверка входных данных

Напишите `validate_input(path, expected_fs=EXPECTED_FS)`:

1. если файла нет, вернуть `Файл {path} не найден.`;
2. если чтение WAV завершилось ошибкой, вернуть `Файл некорректен.`;
3. если частота не равна `expected_fs`, вернуть
   `Частота дискретизации аудио не соответствует модели.`;
4. иначе вернуть `Данные готовы к диагностике`.

In [11]:
# ✍ Задание 5
def validate_input(path, expected_fs=EXPECTED_FS):
    path = Path(path)
    if not path.is_file():
        return f"Файл {path} не найден."
    try:
        sample_rate, signal = wavfile.read(path)
    except Exception:
        return "Файл некорректен."
    if sample_rate != expected_fs:
        return "Частота дискретизации аудио не соответствует модели."
    return "Данные готовы к диагностике"

ans5 = validate_input(DATA / "Подшипник_0.wav")
print(ans5)

Файл data/Подшипник_0.wav не найден.


In [12]:
check(5, ans5)

✅ Задание 5 выполнено


## Задание 6. Повреждённый файл

Вызовите `validate_input` для `Подшипник_.wav` и сохраните результат в `ans6`.

In [13]:
# ✍ Задание 6
ans6 = validate_input(DATA / "Подшипник_.wav")
print(ans6)

Файл некорректен.


In [14]:
check(6, ans6)

✅ Задание 6 выполнено


## Задание 7. Неверная частота дискретизации

Вызовите `validate_input` для `Подшипник_чд.wav` и сохраните результат в `ans7`.

In [15]:
# ✍ Задание 7
ans7 = validate_input(DATA / "Подшипник_чд.wav")
print(ans7)

Частота дискретизации аудио не соответствует модели.


In [16]:
check(7, ans7)

✅ Задание 7 выполнено


## Задание 8. Корректный файл

Вызовите `validate_input` для `Подшипник_1.wav` и сохраните результат в `ans8`.

In [17]:
# ✍ Задание 8
ans8 = validate_input(DATA / "Подшипник_1.wav")
print(ans8)

Данные готовы к диагностике


In [18]:
check(8, ans8)

✅ Задание 8 выполнено


## Итог

Готовая проверка отличает отсутствие файла, повреждённое содержимое и несовместимую
частоту дискретизации. Благодаря этому диагностический расчёт запускается только на
данных, соответствующих ожидаемому формату.